In [3]:
#!/usr/bin/env python3
"""
Gene-by-gene comparison of the two blocks of each patient (same ENSG ID, block A vs block B).
Uses the raw counts matrix; counts are converted to CPM because the two blocks have different
sequencing depth (raw counts of different libraries are not directly comparable).

Per pair we report
  - Spearman / Pearson on log2(CPM+1) over genes expressed in either block
  - genes detected (count > 0) in both blocks / only block A / only block B
  - number and fraction of expressed genes with |log2 fold change| > 1 (block B vs block A)
  - the same Spearman for random pairs of blocks from DIFFERENT patients (reference, seed 42)
Outputs (OUTDIR): gene_pair_summary.csv, gene_counts_pairs.csv (every gene, both counts, CPM, log2FC),
gene_compare_scatter.png/.pdf
"""
import os, re, gzip
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

# ----------------------------- PATHS (edit here) -----------------------------
COUNTS_FILE = "/ix1/alee/LO_LAB/Personal/Rahul/Okwin/RNASeq/counts/matrix/gene_counts.tsv"
MANIFEST    = "/ix1/alee/LO_LAB/Personal/Rahul/Okwin/Owkin_project_manifest.xlsx"
GTF         = ""    # optional: adds a gene-symbol column (same GTF as in duplicate_block_check.py)
OUTDIR      = "/ix1/alee/LO_LAB/Personal/Rahul/Okwin/RNASeq/block_check"
# -----------------------------------------------------------------------------
SEED = 42
rng = np.random.default_rng(SEED)
os.makedirs(OUTDIR, exist_ok=True)

# ---- counts: rows = ENSG IDs, columns = blocks (PI_R_044a_mRNA -> PI_R_044a) ----
raw = pd.read_csv(COUNTS_FILE, sep="\t", index_col=0)
counts = raw.loc[:, raw.columns.str.match(r"^PI_R_\d+[a-z]?(_mRNA)?$")].astype(float)
counts.columns = counts.columns.str.replace(r"_mRNA$", "", regex=True)
cpm = counts / counts.sum(axis=0) * 1e6

# ---- optional gene symbols ----
symbol = None
if GTF:
    op = gzip.open if GTF.endswith(".gz") else open
    mp = {}
    with op(GTF, "rt") as fh:
        for line in fh:
            if line.startswith("#"): continue
            q = line.split("\t")
            if len(q) > 8 and q[2] == "gene":
                gid = re.search(r'gene_id "([^"]+)"', q[8]); gn = re.search(r'gene_name "([^"]+)"', q[8])
                if gid and gn: mp[gid.group(1).split(".")[0]] = gn.group(1)
    symbol = pd.Series({g: mp.get(g.split(".")[0], "") for g in counts.index})

# ---- pairs from the manifest ----
meta = pd.read_excel(MANIFEST, sheet_name="RNAseq", header=2).dropna(subset=["Block ID"]).set_index("Block ID")
meta = meta.loc[[b for b in counts.columns if b in meta.index]]
pairs = [g.index.tolist() for _, g in meta.groupby("Patient") if len(g) == 2]
print(f"Blocks: {counts.shape[1]} | genes: {counts.shape[0]} | patients with two blocks: {len(pairs)}")

def compare(a, b):
    """Statistics for one pair of blocks over genes expressed (CPM > 1) in either block."""
    expr = (cpm[a] > 1) | (cpm[b] > 1)
    la, lb = np.log2(cpm.loc[expr, a] + 1), np.log2(cpm.loc[expr, b] + 1)
    lfc = lb - la
    return expr, la, lb, lfc

rows, long_tables = [], []
for a, b in pairs:
    expr, la, lb, lfc = compare(a, b)
    pat = meta.loc[a, "Patient"]
    # reference: Spearman of random pairs of blocks from different patients
    ref = []
    for _ in range(200):
        x, y = rng.choice(meta.index, 2, replace=False)
        if meta.loc[x, "Patient"] != meta.loc[y, "Patient"]:
            _, lx, ly, _ = compare(x, y)
            ref.append(spearmanr(lx, ly)[0])
    rows.append(dict(
        patient=pat, block_a=a, block_b=b, type_a=meta.loc[a, "Block type"], type_b=meta.loc[b, "Block type"],
        total_counts_a=int(counts[a].sum()), total_counts_b=int(counts[b].sum()),
        genes_expressed_either=int(expr.sum()),
        detected_both=int(((counts[a] > 0) & (counts[b] > 0)).sum()),
        detected_only_a=int(((counts[a] > 0) & (counts[b] == 0)).sum()),
        detected_only_b=int(((counts[a] == 0) & (counts[b] > 0)).sum()),
        spearman_log2cpm=round(spearmanr(la, lb)[0], 3),
        pearson_log2cpm=round(pearsonr(la, lb)[0], 3),
        genes_abs_log2fc_gt1=int((lfc.abs() > 1).sum()),
        fraction_abs_log2fc_gt1=round(float((lfc.abs() > 1).mean()), 3),
        spearman_random_other_patients_median=round(float(np.median(ref)), 3)))
    t = pd.DataFrame({"patient": pat, "Geneid": counts.index,
                      "count_a": counts[a].values, "count_b": counts[b].values,
                      "cpm_a": cpm[a].round(3).values, "cpm_b": cpm[b].round(3).values})
    t["log2FC_b_vs_a"] = np.log2((t.cpm_b + 1) / (t.cpm_a + 1)).round(3)
    t["block_a"], t["block_b"] = a, b
    if symbol is not None: t.insert(2, "symbol", symbol.values)
    long_tables.append(t[(t.count_a > 0) | (t.count_b > 0)])        # drop genes with zero in both
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(OUTDIR, "gene_pair_summary.csv"), index=False)
pd.concat(long_tables).to_csv(os.path.join(OUTDIR, "gene_counts_pairs.csv"), index=False)
print(summary.drop(columns=["total_counts_a", "total_counts_b"]).to_string(index=False))

# ---- scatter grid: log2(CPM+1) block A vs block B for every pair ----
n = len(pairs); ncol = 3; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 4.0 * nrow), squeeze=False)
for ax, (a, b), r in zip(axes.ravel(), pairs, rows):
    _, la, lb, _ = compare(a, b)
    ax.scatter(la, lb, s=3, alpha=.25, c="#2c6fbb", lw=0, rasterized=True)
    m = max(la.max(), lb.max()); ax.plot([0, m], [0, m], c="k", lw=.7)
    ax.plot([0, m], [1, m + 1], c="#a4243b", lw=.5, ls="--"); ax.plot([0, m], [-1, m - 1], c="#a4243b", lw=.5, ls="--")
    ax.set_title(f"{r['patient']}: {r['type_a']} vs {r['type_b']}", fontsize=9)
    ax.set_xlabel(f"{a}  log2(CPM+1)", fontsize=8); ax.set_ylabel(f"{b}  log2(CPM+1)", fontsize=8)
    ax.text(.04, .93, f"Spearman {r['spearman_log2cpm']}\n|log2FC|>1: {100*r['fraction_abs_log2fc_gt1']:.0f}% of genes",
            transform=ax.transAxes, fontsize=8, va="top")
for ax in axes.ravel()[n:]: ax.axis("off")
plt.tight_layout()
for ext in ("png", "pdf"): fig.savefig(os.path.join(OUTDIR, f"gene_compare_scatter.{ext}"), dpi=300)
print("Outputs in", OUTDIR)

Blocks: 84 | genes: 77621 | patients with two blocks: 9
 patient   block_a   block_b  type_a     type_b  genes_expressed_either  detected_both  detected_only_a  detected_only_b  spearman_log2cpm  pearson_log2cpm  genes_abs_log2fc_gt1  fraction_abs_log2fc_gt1  spearman_random_other_patients_median
PI_R_048 PI_R_048a PI_R_048b Primary Metastasis                   20608          28524             7971             4682             0.859            0.872                  6529                    0.317                                  0.859
PI_R_077 PI_R_077a PI_R_077b Primary    Primary                   22093          35424             4494             7372             0.912            0.923                  3577                    0.162                                  0.860
PI_R_090 PI_R_090a PI_R_090b Primary Metastasis                   22231          35838             5851             6189             0.884            0.893                  4512                    0.203                